In [1]:
from pathlib import Path
import os
import time
import random
import threading
import gc

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

from data_helpers import encode_sex_to_01, filter_band, scale_support_table

#### Create saving folder

In [3]:
DATADIR = Path("./data_processed_new")
DATADIR.mkdir(parents=True, exist_ok=True)

#### Load and inspect data

Dataset available upon reasonable request and corresponding authorization. The datas csv file is not uploaded in github

In [4]:
DATA_PATH = Path("./MetClinPar_no_weight_bmi.csv")  # adjust if needed
df = pd.read_csv(DATA_PATH)

# Clean column names
df.columns = (
    df.columns.astype(str)
    .str.replace("\ufeff", "", regex=False)
    .str.strip()
)

# Ensure required targets exist
for col in ["age", "MetSCORE", "sex"]:
    if col not in df.columns:
        raise KeyError(f"Column '{col}' not found in dataset.")

In [5]:
df.head()

,Unnamed: 0,"1,5-Anhydrosorbitol (mM)",2-Aminobutyric acid (mM),2-Hydroxybutyric acid (mM),2-Oxoglutaric acid (mM),3-Hydroxybutyric acid (mM),3-Hydroxyisobutyric acid (mM),Acetic acid (mM),Acetoacetic acid (mM),Acetone (mM),...,Platelets (G/l),GlycB to GlycA ratio (unitless),Non-albumin protein (g/dl),Urea to creatinine ratio (unitless),BCAA to AAA (Phe+Tyr) ratio (unitless),Lactate to pyruvate ratio (unitless),Glutamine to glutamate ratio (unitless),age,sex,MetSCORE
0,22S37180,0.124703,0.060997,0.029497,0.016397,0.043535,0.013533,0.036656,0.015813,0.044793,...,236.47015,0.226339,2.42441,36.607333,4.081604,135.793410,7.466504,56.0,male,0.101325
1,22S17366,0.174066,0.050956,0.024472,0.023113,0.105822,0.022228,0.073641,0.030667,0.038486,...,205.98067,0.231948,2.26718,27.981328,3.849630,112.322106,11.561261,25.0,male,0.053270
2,AAA3693419,0.167167,0.068384,0.071419,0.018260,0.186540,0.021666,0.038915,0.067263,0.087800,...,217.91718,0.231088,1.99292,36.938627,4.466170,83.835707,9.246287,35.0,male,0.013762
3,22S26594,0.115784,0.058953,0.066709,0.017764,0.091046,0.032580,0.033321,0.024105,0.021123,...,232.26855,0.221529,2.31383,42.734883,4.299060,66.784661,8.900252,49.0,male,0.399889
4,17S06171,0.173861,0.076727,0.031121,0.018852,0.032779,0.024747,0.017168,0.016526,0.017583,...,276.53613,0.236253,2.46860,32.595438,4.529253,54.867616,6.998594,43.0,male,0.130309


#### Encode sex as categorical

In [6]:
df = df.copy()
df["sex_num"] = encode_sex_to_01(df["sex"])

#### Drop unused columns

In [7]:
OUTPUT_COLS_NUM = ["age", "MetSCORE", "sex_num"]

# Columns to always drop from inputs
DROP_ALWAYS = {"Unnamed: 0", "index", "ID", "Id", "patient_id", "PatientID"}

# Inputs = numeric columns excluding outputs
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
INPUT_COLS = [c for c in numeric_cols if c not in OUTPUT_COLS_NUM and c not in DROP_ALWAYS]

print("Number of INPUT_COLS:", len(INPUT_COLS))
print("Example INPUT_COLS:", INPUT_COLS[:10])

df_work = df[INPUT_COLS + OUTPUT_COLS_NUM].dropna().reset_index(drop=True)
print("After dropna:", df_work.shape)

Number of INPUT_COLS: 75
Example INPUT_COLS: ['1,5-Anhydrosorbitol (mM)', '2-Aminobutyric acid (mM)', '2-Hydroxybutyric acid (mM)', '2-Oxoglutaric acid (mM)', '3-Hydroxybutyric acid (mM)', '3-Hydroxyisobutyric acid (mM)', 'Acetic acid (mM)', 'Acetoacetic acid (mM)', 'Acetone (mM)', 'Alanine (mM)']
After dropna: (15931, 78)


#### Train - validation - test split --> dataframes

In [8]:
SEED = 1234

train_df, temp_df = train_test_split(df_work, test_size=0.2, random_state=SEED, shuffle=True)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=SEED, shuffle=True)

#### Correlation matrices (train split only)

In [18]:
X_corr = train_df[INPUT_COLS]
corr_inputs = X_corr.corr(method="pearson")
kappa = np.linalg.cond(corr_inputs.values)
print(f"Conditioning before pruning: {kappa}")

corr_inputs_path = DATADIR / "dataframes/corr_inputs_pearson.csv"

Conditioning before pruning: 265.0783349947861


In [19]:
# %% Ranking of highly correlated INPUT pairs by bands -> save .csv
C = corr_inputs.copy()
cols = C.columns.tolist()

pairs = []
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = C.iat[i, j]
        if pd.isna(r):
            continue
        pairs.append((cols[i], cols[j], float(r), abs(float(r))))

pairs_df = pd.DataFrame(pairs, columns=["feature_1", "feature_2", "corr", "abs_corr"])
pairs_df = pairs_df.sort_values("abs_corr", ascending=False).reset_index(drop=True)

band_07_08 = filter_band(pairs_df, 0.7, 0.8, include_hi=False)
band_08_09 = filter_band(pairs_df, 0.8, 0.9, include_hi=False)
band_09_10 = filter_band(pairs_df, 0.9, 1.0, include_hi=True)

In [20]:
FIG_PRE = DATADIR / "figs"
TAB_PRE = DATADIR / "tables"
FIG_PRE.mkdir(parents=True, exist_ok=True)
TAB_PRE.mkdir(parents=True, exist_ok=True)

In [21]:
# --- input-output correlations (Pearson) for age, MetSCORE, sex
Y_targets = train_df[["age", "MetSCORE", "sex_num"]]
corr_in_out = train_df[INPUT_COLS].corrwith(Y_targets["age"]).to_frame("corr_with_age")
corr_in_out["corr_with_MetSCORE"] = train_df[INPUT_COLS].corrwith(Y_targets["MetSCORE"])
corr_in_out["corr_with_sex"] = train_df[INPUT_COLS].corrwith(Y_targets["sex_num"])
corr_in_out["abs_sum"] = corr_in_out[["corr_with_age", 
                                      "corr_with_MetSCORE", 
                                      "corr_with_sex"]].abs().sum(axis=1)

corr_in_out_path = TAB_PRE / "corr_input_output_pearson.csv"
corr_in_out.sort_values("abs_sum", ascending=False).to_csv(corr_in_out_path, float_format="%.6f")

#### Detect correlated features

In [22]:
# --- automatic pruning by input-input abs corr threshold (>=0.85)
THR_CORR = 0.85
corr_inputs_abs = corr_inputs.abs().copy()
np.fill_diagonal(corr_inputs_abs.values, 0.0)

relevance = corr_in_out["abs_sum"].to_dict()

kept = set(INPUT_COLS)
dropped = []
pairs_considered = []

pairs_sorted = []
cols = corr_inputs_abs.columns.tolist()
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        r = corr_inputs_abs.iat[i, j]
        if pd.isna(r):
            continue
        if r >= THR_CORR:
            pairs_sorted.append((cols[i], cols[j], float(r)))
pairs_sorted.sort(key=lambda t: -t[2])

for f1, f2, r in pairs_sorted:
    if (f1 not in kept) or (f2 not in kept):
        continue

    s1 = float(relevance.get(f1, 0.0))
    s2 = float(relevance.get(f2, 0.0))

    if s1 > s2:
        drop = f2
    elif s2 > s1:
        drop = f1
    else:
        # tie-breaker: drop the more redundant one (higher avg abs corr)
        avg1 = float(corr_inputs_abs.loc[f1, list(kept)].mean())
        avg2 = float(corr_inputs_abs.loc[f2, list(kept)].mean())
        drop = f1 if avg1 >= avg2 else f2

    kept.remove(drop)
    dropped.append(drop)
    pairs_considered.append((f1, f2, r, s1, s2, drop))

kept = sorted(list(kept))
dropped = sorted(list(set(dropped)))

In [23]:
prune_df = pd.DataFrame(
    pairs_considered,
    columns=["feature_1", "feature_2", "abs_corr", "rel_sum_f1", "rel_sum_f2", "dropped_feature"]
)
prune_df_path = TAB_PRE / f"pruning_pairs_thr_{THR_CORR:.2f}.csv"
# prune_df.to_csv(prune_df_path, index=False, float_format="%.6f")
print("Saved:", prune_df_path)

(Path(TAB_PRE / f"kept_features_thr_{THR_CORR:.2f}.txt")).write_text("\n".join(kept))
(Path(TAB_PRE / f"dropped_features_thr_{THR_CORR:.2f}.txt")).write_text("\n".join(dropped))
print(f"Pruning summary @ thr={THR_CORR:.2f}: kept={len(kept)} dropped={len(dropped)}")

Saved: data_processed_new/tables/pruning_pairs_thr_0.85.csv
Pruning summary @ thr=0.85: kept=72 dropped=3


In [25]:
support_df = scale_support_table(df, INPUT_COLS)
support_path = TAB_PRE / "scale_support_inputs.csv"
# support_df.to_csv(support_path, float_format="%.6f")
print("Saved:", support_path)

Saved: data_processed_new/tables/scale_support_inputs.csv


#### Remove correlated features

In [26]:
# -----------------------------
# Reproducibility / device
# -----------------------------
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------
# Memory/time profiling utilities
# -----------------------------
try:
    import psutil  # recommended for RSS memory
    _HAS_PSUTIL = True
except Exception:
    psutil = None
    _HAS_PSUTIL = False

In [27]:
# -----------------------------
# Use auto-pruned features from PRE chunk
# -----------------------------
INPUT_COLS_PRUNED = kept
print("N INPUT_COLS_PRUNED:", len(INPUT_COLS_PRUNED))

# -----------------------------
# Model dataframe
# -----------------------------
df_model_train = train_df[INPUT_COLS_PRUNED + ["age", 
                                               "MetSCORE", 
                                               "sex_num"]].dropna().reset_index(drop=True)

df_model_val = val_df[INPUT_COLS_PRUNED + ["age", 
                                           "MetSCORE", 
                                           "sex_num"]].dropna().reset_index(drop=True)

df_model_test = test_df[INPUT_COLS_PRUNED + ["age", 
                                             "MetSCORE", 
                                             "sex_num"]].dropna().reset_index(drop=True)

N INPUT_COLS_PRUNED: 72


In [31]:
X_corr_after = train_df[INPUT_COLS_PRUNED]
corr_inputs_after = X_corr_after.corr(method="pearson")
kappa_after = np.linalg.cond(corr_inputs_after.values)
print(f"Conditioning after pruning: {kappa_after}")

Conditioning after pruning: 158.3141922235487


In [43]:
import pickle
    
with open(DATADIR / "dataframes/input_cols_pruned.pkl", "wb") as f:
    pickle.dump(INPUT_COLS_PRUNED, f)

In [29]:
X_train = df_model_train[INPUT_COLS_PRUNED].values.astype(np.float32)
X_val = df_model_val[INPUT_COLS_PRUNED].values.astype(np.float32)
X_test = df_model_test[INPUT_COLS_PRUNED].values.astype(np.float32)

age_train = df_model_train["age"].values.astype(np.float32).reshape(-1, 1)             
age_val = df_model_val["age"].values.astype(np.float32).reshape(-1, 1)             
age_test = df_model_test["age"].values.astype(np.float32).reshape(-1, 1)             

mets_train = df_model_train["MetSCORE"].values.astype(np.float32).reshape(-1, 1)
mets_val = df_model_val["MetSCORE"].values.astype(np.float32).reshape(-1, 1)
mets_test = df_model_test["MetSCORE"].values.astype(np.float32).reshape(-1, 1)

sex_train = df_model_train["sex_num"].values.astype(np.float32).reshape(-1, 1)
sex_val = df_model_val["sex_num"].values.astype(np.float32).reshape(-1, 1)
sex_test = df_model_test["sex_num"].values.astype(np.float32).reshape(-1, 1)

## SAVE
# np.savez(DATADIR / "train_data.npz",
#          x=X_train, age=age_train, mets=mets_train, sex=sex_train)

# np.savez(DATADIR / "val_data.npz",
#          x=X_val, age=age_val, mets=mets_val, sex=sex_val)

# np.savez(DATADIR / "test_data.npz",
#          x=X_test, age=age_test, mets=mets_test, sex=sex_test)

In [41]:
# -----------------------------
# Normalize inputs + age + MetSCORE (sex stays 0/1)
# -----------------------------
x_mean = X_train.mean(axis=0, keepdims=True)
x_std = X_train.std(axis=0, keepdims=True) + 1e-8
X_train_s = (X_train - x_mean) / x_std
X_val_s = (X_val - x_mean) / x_std
X_test_s = (X_test - x_mean) / x_std

age_mean = age_train.mean(axis=0, keepdims=True)
age_std = age_train.std(axis=0, keepdims=True) + 1e-8
age_train_s = (age_train - age_mean) / age_std
age_val_s = (age_val - age_mean) / age_std
age_test_s = (age_test - age_mean) / age_std

mets_mean = mets_train.mean(axis=0, keepdims=True)
mets_std = mets_train.std(axis=0, keepdims=True) + 1e-8
mets_train_s = (mets_train - mets_mean) / mets_std
mets_val_s = (mets_val - mets_mean) / mets_std
mets_test_s = (mets_test - mets_mean) / mets_std

# np.savez(DATADIR / "scalers.npz",
#     x_mean=x_mean, x_std=x_std,
#     age_mean=age_mean, age_std=age_std,
#     mets_mean=mets_mean, mets_std=mets_std)

# np.savez(DATADIR / "train_data_scaled.npz",
#          x=X_train_s, age=age_train_s, mets=mets_train_s, sex=sex_train)

# np.savez(DATADIR / "val_data_scaled.npz",
#          x=X_val_s, age=age_val_s, mets=mets_val_s, sex=sex_val)

# np.savez(DATADIR / "test_data_scaled.npz",
#          x=X_test_s, age=age_test_s, mets=mets_test_s, sex=sex_test)